# Projection Summary

Consolidated summary of SSP CISI projections from projections_delta.ipynb.
Run after projections_delta.ipynb has been executed (reuses its output files).

In [ ]:
import os
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import geopandas as gpd


## Setup

In [ ]:
PRED_DIR   = r'READY_data/predictions'
CISI_PATH  = r'READY_data/labels/2024_CISI_010deg_nearest.tif'
SHP_PATH   = r'ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp'
SSPS       = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
YEARS      = [2030, 2050, 2100]

def load(path):
    with rasterio.open(path) as src:
        d = src.read(1).astype(np.float32)
        transform = src.transform
        crs = src.crs
    d[d < -1e10] = np.nan
    d[np.isinf(d)] = np.nan
    return d, transform, crs

cisi_2024, transform, crs = load(CISI_PATH)

results = {}
for ssp in SSPS:
    results[ssp] = {}
    for year in YEARS:
        path = os.path.join(PRED_DIR, 'delta_projected_' + ssp + '_' + str(year) + '.tif')
        if os.path.exists(path):
            results[ssp][year], _, _ = load(path)
        else:
            print('Missing: ' + path)

print('Loaded predictions for:', list(results.keys()))


## Summary statistics table

In [ ]:
print('SSP'.ljust(6) + 'Year'.rjust(6) + 'Mean proj'.rjust(12) + 'Mean delta'.rjust(12)
      + 'Max proj'.rjust(10) + 'High>0.15(%)'.rjust(14))
print('-' * 60)
for ssp in SSPS:
    for year in YEARS:
        if year not in results.get(ssp, {}):
            continue
        proj  = results[ssp][year]
        delta = proj - cisi_2024
        valid = proj[~np.isnan(proj)]
        high  = (valid > 0.15).mean() * 100
        print(ssp.ljust(6) + str(year).rjust(6)
              + str(round(float(valid.mean()), 4)).rjust(12)
              + ('+' + str(round(float(np.nanmean(delta)), 4)) if np.nanmean(delta) >= 0
                 else str(round(float(np.nanmean(delta)), 4))).rjust(12)
              + str(round(float(valid.max()), 4)).rjust(10)
              + (str(round(high, 1)) + '%').rjust(14))


## Mean CISI per SSP over time

In [ ]:
COLORS = {'SSP1': '#4daf4a', 'SSP2': '#377eb8', 'SSP3': '#ff7f00',
          'SSP4': '#984ea3', 'SSP5': '#e41a1c'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Mean projected CISI per SSP scenario', fontsize=13)

ax1, ax2 = axes

for ssp in SSPS:
    means  = [float(np.nanmean(results[ssp][y])) for y in YEARS if y in results[ssp]]
    deltas = [float(np.nanmean(results[ssp][y] - cisi_2024)) for y in YEARS if y in results[ssp]]
    yrs    = [y for y in YEARS if y in results[ssp]]
    ax1.plot(yrs, means,  marker='o', color=COLORS[ssp], linewidth=2, label=ssp)
    ax2.plot(yrs, deltas, marker='o', color=COLORS[ssp], linewidth=2, label=ssp)

# 2024 baseline
baseline = float(np.nanmean(cisi_2024))
ax1.axhline(baseline, color='black', linestyle='--', linewidth=1.5, label='2024 observed')
ax2.axhline(0, color='black', linestyle='--', linewidth=1.5, label='No change')

ax1.set_title('Absolute mean CISI')
ax1.set_xlabel('Year'); ax1.set_xticks(YEARS)
ax1.set_ylabel('Mean CISI'); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

ax2.set_title('Delta vs 2024 baseline')
ax2.set_xlabel('Year'); ax2.set_xticks(YEARS)
ax2.set_ylabel('Delta CISI'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/summary_mean_cisi_per_ssp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/summary_mean_cisi_per_ssp.png')


## SSP5 minus SSP1 delta (scenario spread over time)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('SSP5 minus SSP1 -- scenario spread', fontsize=13)

spread_mean = []
spread_std  = []
for year in YEARS:
    diff = results['SSP5'][year] - results['SSP1'][year]
    spread_mean.append(float(np.nanmean(diff)))
    spread_std.append(float(np.nanstd(diff)))

axes[0].bar([str(y) for y in YEARS], spread_mean, color='#e41a1c', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Mean SSP5 - SSP1 CISI difference')
axes[0].set_ylabel('Delta CISI'); axes[0].grid(True, alpha=0.3)

axes[1].bar([str(y) for y in YEARS], spread_std, color='#377eb8', alpha=0.8)
axes[1].set_title('Std of SSP5 - SSP1 pixel differences')
axes[1].set_ylabel('Std CISI'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/summary_ssp_spread.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/summary_ssp_spread.png')


## Note on negative deltas

All SSP projections show slightly negative mean delta vs 2024 (CISI decreasing over time).
This is a known model limitation: the CNN was trained on cross-sectional spatial variation in 2019-2020 data,
not on temporal change. The model maps input patterns to CISI values but cannot learn
how infrastructure grows over time. The small negative drift may reflect that SSP future
population/GDP distributions (even after unit scaling) differ subtly from the training distribution,
causing a slight downward bias in predictions.

The between-scenario differences (SSP5 > SSP1) are real and growing over time, which is the
meaningful result to report.